# Helper notebook for results administration

This notebook helps manage the repository `results/` folder.

It can solve two practical tasks:

1. **Build a parameter-set overview table** to see what parameter set hides behind the hashed result folders names.
2. **Delete outdated result folders safely** to clean `results/` repo when old and new runs become mixed.

Quick `runner.py` logic explanation:

- results are stored under `results/<ModelName>_sweep/params_<hash>/...` or `results/<ModelName>_timecourse/params_<hash>/...`
- the hash belongs to the **effective parameter set**, not to a single run
- each `params_<hash>` folder may contain a `param_set.json` manifest
- each run folder may contain `run_config.json`, `run_summary.json`, and `run.h5`


In [ ]:
# Imports and repository setup

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# -------------------------
# Resolve repository root
# -------------------------
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "scripts" / "runner.py").exists()
            and (candidate / "configs").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate repository root. Please run this notebook from inside the repo "
        "or check that scripts/runner.py, configs/, and models/ exist."
    )

repo_root = find_repo_root()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
from scripts.runner import run_reference, run_one, run_sweep, load_config

RESULTS_DIR = repo_root / 'results'

print(f"results_dir: {RESULTS_DIR}")


## Build the Param-set table
This cell is building an overview table of all folders and parameter-set folders:

- First, choose which models and result kinds to scan.
- Then run the scan cell to build the Param-set table.
- Use the deletion helpers below, if old folders are to be erased.


## Input

**`MODELS`**
Use either:

- `"all"` to scan every detected model under `results/`, or
- a list such as `["EASI", "TARC"]`

**`KINDS`**
Choose which result trees to scan:

- `"sweep"`
- `"timecourse"`

**`EXPAND_DISPLAY`**
- Long entries are truncated if `False`

**`MAX_ROWS`**
- Maximum number of rows that should be displayed



In [ ]:
# -------------------------
# INPUT (edit only here)
# -------------------------

# Choose models:
# - "all" to scan all detected models or a list like ["EASI", "TARC"]
MODELS = "all"

# Choose result kinds:
# - any subset of ["sweep", "timecourse"]
KINDS = ["sweep", "timecourse"]

# Show full text columns without truncation
EXPAND_DISPLAY = False

# Maximum number of rows to display
MAX_ROWS = 200
# -------------------------


def detect_models(results_dir: Path):
    """Detect model names from folders such as TARC_sweep or TARC_timecourse."""
    if not results_dir.exists():
        return []

    models = set()
    for p in results_dir.iterdir():
        if not p.is_dir():
            continue
        name = p.name
        if name.endswith("_sweep"):
            models.add(name[:-6])
        elif name.endswith("_timecourse"):
            models.add(name[:-11])
    return sorted(models)


def normalize_models(results_dir: Path, models):
    """Resolve 'all' into all detected model names."""
    detected = detect_models(results_dir)
    if models == "all":
        return detected
    if isinstance(models, str):
        if models.lower() == "all":
            return detected
        return [models]
    return list(models)


def safe_read_json(path: Path):
    """Read a JSON file safely. Return None on failure."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def compact_dict(d):
    """Convert a dict into a short readable string for table display."""
    if not isinstance(d, dict) or not d:
        return ""
    return ", ".join(f"{k}={v}" for k, v in sorted(d.items()))


def summarize_values(values):
    """Return a sorted compact list for stable display."""
    clean = []
    for v in values:
        if v is None:
            continue
        if v not in clean:
            clean.append(v)
    try:
        return sorted(clean)
    except Exception:
        return clean


def collect_param_set_row(param_root: Path, model: str, kind: str):
    """
    Build one summary row for one params_<hash> folder.
    Uses:
      - param_set.json for parameter metadata
      - run_config.json / run_summary.json / run.h5 presence to summarize runs
    """
    manifest_path = param_root / "param_set.json"
    manifest = safe_read_json(manifest_path)

    manifest_error = manifest is None

    param_overrides = {}
    created_at = None

    if manifest:
        param_overrides = manifest.get("param_overrides", {}) or {}
        created_at = manifest.get("created_at", None)

    run_dirs = [p for p in param_root.iterdir() if p.is_dir()]

    dose_values = []
    interval_values = []
    n_doses_values = []
    n_runs_total = 0
    n_runs_with_summary = 0
    n_runs_with_config = 0
    n_runs_with_h5 = 0

    for run_dir in run_dirs:
        n_runs_total += 1

        summary_path = run_dir / "run_summary.json"
        config_path = run_dir / "run_config.json"
        h5_path = run_dir / "run.h5"

        if summary_path.exists():
            n_runs_with_summary += 1
        if config_path.exists():
            n_runs_with_config += 1
        if h5_path.exists():
            n_runs_with_h5 += 1

        cfg = safe_read_json(config_path)
        if cfg:
            run_params = cfg.get("run_params", {}) or {}
            dose_values.append(run_params.get("dose_mgkg"))
            interval_values.append(run_params.get("interval_weeks"))
            n_doses_values.append(run_params.get("n_doses"))

    return {
        "model": model,
        "kind": kind,
        "params_hash": param_root.name.replace("params_", "", 1),
        "created_at": created_at,
        "n_runs_total": n_runs_total,
        "n_runs_with_summary": n_runs_with_summary,
        "n_runs_with_config": n_runs_with_config,
        "n_runs_with_h5": n_runs_with_h5,
        "dose_values": summarize_values(dose_values),
        "interval_values": summarize_values(interval_values),
        "n_doses_values": summarize_values(n_doses_values),
        "param_overrides": compact_dict(param_overrides),
        "param_root": str(param_root),
        "manifest_error": manifest_error,
    }


def build_param_set_table(results_dir: Path, models, kinds):
    """Scan results folders and build one dataframe with one row per params_<hash> folder."""
    rows = []
    selected_models = normalize_models(results_dir, models)

    for model in selected_models:
        for kind in kinds:
            kind_root = results_dir / f"{model}_{kind}"
            if not kind_root.exists():
                continue

            for param_root in sorted(kind_root.glob("params_*")):
                if not param_root.is_dir():
                    continue
                rows.append(collect_param_set_row(param_root, model, kind))

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    sort_cols = [c for c in ["model", "kind", "created_at", "params_hash"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, na_position="last").reset_index(drop=True)

    return df


if EXPAND_DISPLAY:
    pd.set_option("display.max_colwidth", None)
else:
    pd.set_option("display.max_colwidth", 80)

pd.set_option("display.max_rows", MAX_ROWS)

detected_models = detect_models(RESULTS_DIR)
selected_models = normalize_models(RESULTS_DIR, MODELS)

print("Detected models under results/:", detected_models)
print("Selected models:", selected_models)
print("Selected kinds:", KINDS)

param_df = build_param_set_table(RESULTS_DIR, MODELS, KINDS)

print(f"Param-set rows: {len(param_df)}")
display(param_df)

## Delete result folders

This cell allows safe deletion of result folders in `results/`.  
It is designed to prevent accidental data loss by using preview and confirmation steps.

### Inputs

- **DELETE_MODE**  
  - `'model_kind'`: delete a full folder like `<Model>_sweep` or `<Model>_timecourse`
  - `'param_set'`: delete a single `params_<hash>` folder

- **TARGET_PATH**  
  Path to the folder you want to delete:

  - for `'model_kind'`: Path to a full `<Model>_sweep` or `<Model>_timecourse` folder
      - e.g. `repo_root / 'results' / 'TARC_sweep'`
  
  - for `'param_set'`: Path to a specific `params_<hash>` folder
      - e.g. `repo_root / 'results' / 'TARC_sweep' / 'params_0262be5ea352'`

- **DRY_RUN**  
  If `True`, only shows what would be deleted (recommended default).  
  If `False`, deletion can proceed (with confirmation).

- **CONFIRM**  
  Must be set to `"DELETE"` to execute deletion when `DRY_RUN = False`.


This ensures that deletion is always explicit and intentional.


In [ ]:

# -------------------------
# INPUT (edit only here)
# -------------------------

DELETE_MODE = "param_set / model_kind"  

# If DELETE_MODE='param_set' => TARGET_PATH should point to one params_<hash> folder, e.g.
# If DELETE_MODE='model_kind' => TARGET_PATH should point to one full folder
TARGET_PATH = repo_root / "results" / "REPLACE_ME"

# Safety switches
DRY_RUN = True
CONFIRM = ""   # must be "DELETE" to actually delete

# -------------------------
# HELPER FUNCTIONS
# -------------------------
def normalize_target_path(target_path) -> Path:
    """Resolve TARGET_PATH into an absolute Path, relative to repo_root if needed."""
    p = Path(target_path)
    if not p.is_absolute():
        p = (repo_root / p).resolve()
    return p

def count_contents(path: Path):
    """Return a small summary of how many folders and files exist below a path."""
    n_dirs = 0
    n_files = 0
    if not path.exists():
        return n_dirs, n_files

    for p in path.rglob("*"):
        if p.is_dir():
            n_dirs += 1
        elif p.is_file():
            n_files += 1

    return n_dirs, n_files

def validate_delete_target(path: Path, mode: str):
    """Validate that the target matches the requested deletion mode."""
    if not path.exists():
        raise FileNotFoundError(f"Target does not exist: {path}")

    if not path.is_dir():
        raise ValueError(f"Target must be a directory: {path}")

    if mode == "param_set":
        if not path.name.startswith("params_"):
            raise ValueError(
                'For DELETE_MODE="param_set", TARGET_PATH must be a params_<hash> folder.'
            )

    elif mode == "model_kind":
        if not (path.name.endswith("_sweep") or path.name.endswith("_timecourse")):
            raise ValueError(
                'For DELETE_MODE="model_kind", TARGET_PATH must be a <Model>_sweep or <Model>_timecourse folder.'
            )

    else:
        raise ValueError('DELETE_MODE must be either "param_set" or "model_kind".')

    # Extra safety: prevent catastrophic deletes
    if path == repo_root:
        raise ValueError("Refusing to delete repo_root.")
    if str(path) in ("/", str(Path("/").resolve())):
        raise ValueError("Refusing to delete filesystem root.")

def preview_delete(path: Path, mode: str):
    """Print a compact preview of what would be removed."""
    validate_delete_target(path, mode)
    n_dirs, n_files = count_contents(path)

    print("DELETE PREVIEW")
    print("--------------")
    print("mode      :", mode)
    print("target    :", path)
    print("subfolders:", n_dirs)
    print("files     :", n_files)

def delete_target(path: Path, mode: str, dry_run: bool = True, confirm: str = ""):
    """
    Delete a target only when dry_run is False and confirm == 'DELETE'.
    Always prints preview first.
    """
    preview_delete(path, mode)

    if dry_run:
        print("Dry run only. Nothing was deleted.")
        return

    if confirm != "DELETE":
        raise ValueError('Deletion blocked. Set CONFIRM = "DELETE" to execute.')

    shutil.rmtree(path)
    print("Deletion complete.")

# -------------------------
# MAIN
# -------------------------
TARGET_PATH = normalize_target_path(TARGET_PATH)

# Always show preview; delete only if DRY_RUN is False and CONFIRM is correct
delete_target(TARGET_PATH, DELETE_MODE, dry_run=DRY_RUN, confirm=CONFIRM)